In [ ]:
%%capture
import os
import pandas as pd
from dj_notebook import activate
from pathlib import Path
env_file = os.environ["INTECOMM_ENV"]
analysis_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
reports_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
plus = activate(dotenv_file=env_file)
pd.set_option('future.no_silent_downcasting', True)

In [ ]:
from intecomm_analytics.dataframes import get_hiv_rx_crf, get_htn_rx_crf, get_dm_rx_crf
from intecomm_analytics.dataframes import get_medications_df, get_appt_df


In [ ]:
variable_labels = {}

In [ ]:
def get_medications(df_appt:pd.DataFrame) -> pd.DataFrame:
    df_meds = pd.merge(df_appt, get_dm_rx_crf("dm"), on="appointment_id", how="left", suffixes=("", "_y"))
    df_meds = df_meds.drop(columns = [col for col in df_meds.columns if col.endswith("_y")])

    df_meds = pd.merge(df_meds, get_htn_rx_crf("htn"), on="appointment_id", how="left", suffixes=("", "_y"))
    df_meds = df_meds.drop(columns = [col for col in df_meds.columns if col.endswith("_y")])

    df_meds = pd.merge(df_meds, get_hiv_rx_crf(), on="appointment_id", how="left", suffixes=("", "_y"))
    df_meds = df.drop(columns = [col for col in df_meds.columns if col.endswith("_y")])

    df_meds = df_meds.drop(columns=["appt_type_other", "document_status_comments", "id"])

    df_meds = df_meds.rename({
        "htn_rx_modifications_reason_other": "htn_rx_mod_reason_other",
        "hiv_rx_modifications_reason_other": "hiv_rx_mod_reason_other",
        "dm_rx_modifications_reason_other": "dm_rx_mod_reason_other"
    })

    df_meds["appointment_id"] = df_meds["appointment_id"].astype(str)
    df_meds["subject_visit_id"] = df_meds["subject_visit_id"].astype(str)
    df_meds["reason_unscheduled_other"] = df_meds["reason_unscheduled_other"].astype(str)
    df_meds["hiv_rx_modifications_other"] = df_meds["hiv_rx_modifications_other"].astype(str)
    df_meds["htn_rx_modifications_other"] = df_meds["htn_rx_modifications_other"].astype(str)
    df_meds["dm_rx_modifications_other"] = df_meds["dm_rx_modifications_other"].astype(str)

    df_meds["htn_rx_mod_reason_other"] = df_meds["htn_rx_mod_reason_other"].astype(str)
    df_meds["hiv_rx_mod_reason_other"] = df_meds["hiv_rx_mod_reason_other"].astype(str)
    df_meds["dm_rx_mod_reason_other"] = df_meds["dm_rx_mod_reason_other"].astype(str)

    df_meds["timepoint"] = df_meds["timepoint"].astype("Float64")
    return df_meds

In [ ]:
df_meds, medication_variable_labels = get_medications_df()

In [ ]:
df_appt = get_appt_df()

In [ ]:
suffix = "dm"
df_rx = get_dm_rx_crf(suffix)
df = pd.merge(df_appt, df_rx, on="appointment_id", how="left", suffixes=("", "_y"))
df = df.drop(columns = [col for col in df.columns if col.endswith("_y")])


In [ ]:
suffix = "htn"
df_rx = get_htn_rx_crf(suffix)
df = pd.merge(df, df_rx, on="appointment_id", how="left", suffixes=("", "_y"))
df = df.drop(columns = [col for col in df.columns if col.endswith("_y")])

# df = merge_with_visit(df, df_rx, suffix)

In [ ]:
suffix = "hiv"
df_rx = get_hiv_rx_crf()
df = pd.merge(df, df_rx, on="appointment_id", how="left", suffixes=("", "_y"))
df = df.drop(columns = [col for col in df.columns if col.endswith("_y")])

# df = merge_with_visit(df, df_rx, suffix)

In [ ]:
df = df.drop(columns=["appt_type_other"])

In [ ]:
df["appointment_id"] = df["appointment_id"].astype(str)
df["subject_visit_id"] = df["subject_visit_id"].astype(str)
df["reason_unscheduled_other"] = df["reason_unscheduled_other"].astype(str)
df["hiv_rx_modifications_other"] = df["hiv_rx_modifications_other"].astype(str)
df["htn_rx_modifications_other"] = df["htn_rx_modifications_other"].astype(str)
df["dm_rx_modifications_other"] = df["dm_rx_modifications_other"].astype(str)

df["hiv_rx_modifications_reason_other"] = df["hiv_rx_modifications_reason_other"].astype(str)
df["htn_rx_modifications_reason_other"] = df["htn_rx_modifications_reason_other"].astype(str)
df["dm_rx_modifications_reason_other"] = df["dm_rx_modifications_reason_other"].astype(str)

In [ ]:
df["timepoint"] = df["timepoint"].astype("Float64")


In [ ]:
df = df.drop(columns=["document_status_comments"])


In [ ]:
df = df.drop(columns=["id"])

In [ ]:
df.to_stata(
    path=analysis_folder / "medications_1858.dta",
    #variable_labels=stata_labels,
    version=118,
    write_index=False,
)